In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('police_incidents').getOrCreate()

In [0]:
#Removing duplicates by id from the bronze table
df_spark = spark.table('thesis_police.bronze.incidents').dropDuplicates(['id'])

In [0]:
df_spark.printSchema()

root
 |-- datetime: string (nullable = true)
 |-- id: long (nullable = true)
 |-- location: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- name: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- type: string (nullable = true)
 |-- url: string (nullable = true)



In [0]:
#Transforming the following columns where location is extracted into two separate columns, location_name and location_gps, time is extracted from name, incident type is extracted from name and area as well, incident date is also extracted since the current date column that is available is regarded as published date, and published date is converted to a timestamp.

from pyspark.sql.functions import col, split, regexp_extract, trim, to_timestamp

df_silver = df_spark.select(col('id'),
                            col('summary'),
                            col('type'),
                            col('url'),
                            col('location.name').alias('location_name'),
                            col('location.gps').alias('location_gps'),
                            col('name'),
                            regexp_extract(col('name'), r'(\d{1,2}\.\d{2})', 1).alias('incident_time'),
                            trim(split(col('name'), ',').getItem(1)).alias('incident_type'),
                            trim(split(col('name'), ',').getItem(2)).alias('area_name'),
                            regexp_extract(col('name'), r'^(\d{1,2}\s+\w+)', 1).alias('incident_date'),
                            to_timestamp(col('datetime')).alias('published_time'),
          
)

In [0]:
df_silver.printSchema()

root
 |-- id: long (nullable = true)
 |-- summary: string (nullable = true)
 |-- type: string (nullable = true)
 |-- url: string (nullable = true)
 |-- location_name: string (nullable = true)
 |-- location_gps: string (nullable = true)
 |-- name: string (nullable = true)
 |-- incident_time: string (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- area_name: string (nullable = true)
 |-- incident_date: string (nullable = true)
 |-- published_time: timestamp (nullable = true)



In [0]:
#Reordering the columns
df_silver = df_silver.select('id', 'incident_date', 'incident_time',
                             'incident_type', 'type', 'location_name', 'area_name',
                             'name', 'location_gps', 'summary', 'published_time', 'url') 

In [0]:
df_silver.display()

id,incident_date,incident_time,incident_type,type,location_name,area_name,name,location_gps,summary,published_time,url
615456,30 november,14.34,Trafikolycka,Trafikolycka,Falun,Falun,"30 november 14.34, Trafikolycka, Falun","60.60646,15.6355",En olycka har inträffat i höjd med Vika,2025-11-30T13:46:58.000Z,/aktuellt/handelser/2025/november/30/30-november-14.34-trafikolycka-falun/
615452,30 november,14.07,Trafikolycka,Trafikolycka,Mariestad,Mariestad,"30 november 14.07, Trafikolycka, Mariestad","58.710112,13.821333",En husbil och en personbil är involverad i en olycka på E20 vid Österberga.,2025-11-30T13:27:54.000Z,/aktuellt/handelser/2025/november/30/30-november-14.07-trafikolycka-mariestad/
615445,30 november,12.29,Trafikolycka,Trafikolycka,Borlänge,Borlänge,"30 november 12.29, Trafikolycka, Borlänge","60.484304,15.433969",En olycka har inträffat i Stora Tuna,2025-11-30T11:47:10.000Z,/aktuellt/handelser/2025/november/30/30-november-12.29-trafikolycka-borlange/
615442,30 november,06.47,Misshandel,Misshandel,Piteå,Piteå,"30 november 06.47, Misshandel, Piteå","65.316698,21.480036",Polispatrull beordras till Hortlax med anledning av en konflikt mellan en taxichaufför och en kund gällande be,2025-11-30T10:51:54.000Z,/aktuellt/handelser/2025/november/30/30-november-06.47-misshandel-pitea/
615403,29 november,21.58,Övrigt,Övrigt,Västra Götalands län,Västra Götalands län,"29 november 21.58, Övrigt, Västra Götalands län","58.252793,13.059643",Efter klockan 22:00 finns ingen presstalesperson i tjänst. Frågor från media besvaras av befäl på ledningscent,2025-11-29T20:58:24.000Z,/aktuellt/handelser/2025/november/29/29-november-21.58-ovrigt-vastra-gotalands-lan/
615380,29 november,13.17,Trafikolycka,Trafikolycka,Boden,Boden,"29 november 13.17, Trafikolycka, Boden","65.825119,21.688703","Larm om trafikolycka på Sveavägen/Åbergsleden, Boden.",2025-11-29T12:29:54.000Z,/aktuellt/handelser/2025/november/29/29-november-13.17-trafikolycka-boden/
615348,29 november,07.05,Sammanfattning kväll och natt,Sammanfattning kväll och natt,Västra Götalands län,Västra Götalands län,"29 november 07.05, Sammanfattning kväll och natt, Västra Götalands län","58.252793,13.059643",Dagens presstalesperson är på plats. Här följer ett urval av nattens händelser.,2025-11-29T06:06:15.000Z,/aktuellt/handelser/2025/november/29/29-november-07.05-sammanfattning-kvall-och-natt-vastra-gotalands-lan/
615329,28 november,19.34,Trafikolycka,"Trafikolycka, vilt",Norrbottens län,vilt,"28 november 19.34, Trafikolycka, vilt, Norrbottens län","66.830922,20.399197",Ett antal viltolyckor och renpåkörningar har rapporterats in till polisens ledningscentral under eftermiddagen,2025-11-28T18:36:01.000Z,/aktuellt/handelser/2025/november/28/28-november-19.34-trafikolycka-vilt-norrbottens-lan/
615322,28 november,16.54,Stöld/inbrott,Stöld/inbrott,Stenungsund,Stenungsund,"28 november 16.54, Stöld/inbrott, Stenungsund","58.067839,11.829434",Polisen larmas om ett pågående inbrott i en lägenhet i centrala Stenungsund.,2025-11-28T16:24:39.000Z,/aktuellt/handelser/2025/november/28/28-november-16.54-stoldinbrott-stenungsund/
615215,28 november,07.00,Sammanfattning natt,Sammanfattning natt,Hallands län,Hallands län,"28 november 07.00, Sammanfattning natt, Hallands län","56.896681,12.803399",Dagens presstalesperson är på plats.,2025-11-28T06:03:56.000Z,/aktuellt/handelser/2025/november/28/28-november-07.00-sammanfattning-natt-hallands-lan/


In [0]:
#Dropping the columns that are not needed
df_silver = df_silver.drop('area_name', 'type')

In [0]:
df_silver.printSchema()

root
 |-- id: long (nullable = true)
 |-- incident_date: string (nullable = true)
 |-- incident_time: string (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- location_name: string (nullable = true)
 |-- name: string (nullable = true)
 |-- location_gps: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- published_time: timestamp (nullable = true)
 |-- url: string (nullable = true)



In [0]:
df_silver.count()

4780

In [0]:
#Dropping duplicates based on id
df_silver = df_silver.dropDuplicates(['id'])

In [0]:
#Saving the data to the silver table
df_silver.write.format('delta').mode('overwrite').saveAsTable('thesis_police.silver.incidents_clean')